In [29]:
from dataclasses import dataclass
import os
from pathlib import Path

## Entity

In [30]:
@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [31]:
os.getcwd()

'd:\\AI-Food-Recognition-Nutrition-Assistant'

In [6]:
os.chdir("..")

In [18]:
%pwd

'd:\\AI-Food-Recognition-Nutrition-Assistant'

## Configuration Manager

In [ ]:
from AI_Food_Recognition_Nutrition_Assistant.constants import *
from AI_Food_Recognition_Nutrition_Assistant.utils.common import read_yaml,create_directories,decodeImage

In [37]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
        root_dir = config.root_dir,
        source_url = config.source_url,
        local_data_file = config.local_data_file,
        unzip_dir = config.unzip_dir,   
        )

        return data_ingestion_config

## Data Ingestion Component

In [38]:
import zipfile
import urllib.request as request
from AI_Food_Recognition_Nutrition_Assistant import logger
from AI_Food_Recognition_Nutrition_Assistant.utils.common import get_size

In [39]:
class DataIngestion:
    
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
                )
            logger.info(f"{filename} downloaded successfully with info {headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")


    def extract_zipfile(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

## Pipeline

In [40]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zipfile()
except Exception as e:
    raise e

[2026-03-13 16:59:07,505: INFO: common: yaml file: <_io.TextIOWrapper name='config\\config.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-13 16:59:07,506: INFO: common: yaml file: <_io.TextIOWrapper name='params.yaml' mode='r' encoding='cp1252'> loaded successfully]
[2026-03-13 16:59:07,508: INFO: common: created directory at: artifacts]
[2026-03-13 16:59:07,509: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-13 17:03:47,050: INFO: 3471712837: artifacts/data_ingestion/data.zip downloaded successfully with info Content-Type: application/zip
X-GUploader-UploadID: AGQBYWyn3laG6kHzV2yQfCCOoYvC11Z9QNw6xCbl5mQb__4cwia-rneDQ30bTEUvP4M0URcP
Expires: Fri, 13 Mar 2026 15:59:09 GMT
Date: Fri, 13 Mar 2026 15:59:09 GMT
Cache-Control: private, max-age=0
Last-Modified: Thu, 21 Nov 2019 01:36:33 GMT
ETag: "4c697f34e0f1b5db4b95cb3793c02592"
x-goog-generation: 1574300193295556
x-goog-metageneration: 1
x-goog-stored-content-encoding: identity
x-goog-stored-content